# QZAP Text Analysis: Data Pipeline

## About This Notebook

This notebook builds a text analysis pipeline for a curated sample of zines from the Queer Zine Archive Project (QZAP), a free digital archive of queer zines available at https://www.qzap.org.

Zines were identified using the catalog built in `01_data_access.ipynb`, downloaded as PDFs, and saved to `data/zines/`. This notebook picks up from there: it loads those files, extracts their text, cleans and tokenizes it, and runs word frequency and TF-IDF analysis.

The selection process required two rounds. My first set of 10 zines included 3 that turned out to be unextractable — one corrupt file and two image-based scans with no text layer. I identified these failures through extraction, then replaced them with 3 new zines from similar time periods. The full process is documented below.

## Research Questions

1. What words and phrases most distinctively characterize individual zines in this sample?
2. How does the vocabulary of each zine reflect its community, politics, and moment?

The first question is addressed through word frequency analysis: a simple count of the most common words across all ten zines combined. This gives a baseline picture of the shared vocabulary in the archive.

The second and primary question is addressed through TF-IDF (Term Frequency–Inverse Document Frequency), which identifies words that are distinctive to each individual zine.

## Selecting Zines for Analysis

The catalog built in `01_data_access.ipynb` contains all 532 digitized zines in QZAP, saved to `output/qzap_catalog.csv`. From that catalog, I selected 10 zines using the cells below. My selection criteria were:

1. **Decade spread**: I wanted representation across the archive's full date range, not just recent zines.
2. **Legibility**: I browsed each candidate zine by hand before selecting it, prioritizing digitally typeset pages over handwritten or heavily image-based layouts.
3. **Thematic variety**: I tried to select zines that address different topics, communities, and styles, so TF-IDF has meaningful contrast to work with.

However, legibility is difficult to judge from a thumbnail alone. The real test is whether `pdfplumber` can extract usable text from the PDF. This section documents the full process, including the first attempt, the failures, and the replacements.

In [ ]:
import pdfplumber
import pandas as pd
import matplotlib.pyplot as plt
import os
import re
import time
import requests
from string import punctuation
from bs4 import BeautifulSoup
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer

# Download nltk data if needed (only needs to run once)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)

stop_words = set(stopwords.words("english"))
headers = {"User-Agent": "Mozilla/5.0"}

print("Libraries loaded!")
print("Stop words count:", len(stop_words))

**Note:** This cell takes approximately 25 minutes to run and only needs to be 
run once. The output is already saved to `output/qzap_catalog.csv` — skip this 
cell and load the CSV directly using the cell below.

In [ ]:
# Load the pre-built catalog saved by 01_data_access.ipynb
catalog_df = pd.read_csv("../output/qzap_catalog.csv")
print("Catalog loaded:", len(catalog_df), "zines")

# Browse a random sample of 20 zines from 1990 onward to find candidates
catalog_df["year_clean"] = pd.to_numeric(catalog_df["year"], errors="coerce")
candidates = catalog_df[catalog_df["year_clean"] >= 1990].sample(20, random_state=42)
candidates[["object_id", "title", "year", "download_url"]]

In [ ]:
# First selection of 10 zines for analysis
selected_titles = [
    "Behind The Bars",
    "Empty Orchestra #1",
    "We Have Always Existed: Transgender People through History",
    "Easily Grossed Out #2",
    "Going Homo #3",
    "PATS #23",
    "Gender Fuck Me",
    "Queer Zine Explosion #20",
    "The Anarchistic Queer #1.1",
    "Gaysi #1"
]

selected = catalog_df[catalog_df["title"].isin(selected_titles)]
print("Found:", len(selected), "zines in catalog")
selected[["title", "year", "download_url"]]

## Downloading the Zines

The function `download_zine_auto()` was originally defined in `01_data_access.ipynb`. I redefine it here so this notebook is self-contained and can be run independently.

Given a download URL from the catalog, the function:
1. Extracts the `object_id` from the URL
2. Fetches the zine's page on QZAP to read its title and year
3. Builds a clean filename in the format `title_year.pdf`
4. Downloads the PDF and saves it to `data/zines/`

I include a 1-second delay between requests to avoid overloading QZAP's servers.

In [ ]:
def download_zine_auto(download_url):
    """
    Given a QZAP download URL, automatically extracts title and year
    from the zine page, then downloads and saves the PDF to data/zines/.
    """
    # Extract object_id from the download URL
    object_id = re.search(r"object_id/(\d+)", download_url).group(1)

    # Fetch the zine's page to get its title and year
    page_url = f"https://archive.qzap.org/index.php/Detail/Object/Show/object_id/{object_id}"
    page = requests.get(page_url, headers=headers)
    soup = BeautifulSoup(page.content, "html.parser")

    # Extract title and year from the page
    title = soup.find("h1").get_text().replace("Zine: ", "").strip()
    year_match = re.search(r"Date:\s*(\d{4})", soup.get_text())
    year = year_match.group(1) if year_match else "unknown"

    # Build a clean filename
    filename = title.lower().replace(" ", "_") + f"_{year}.pdf"
    filename = re.sub(r'[^\w\-_.]', '', filename)

    # Download and save the PDF
    response = requests.get(download_url, headers=headers)
    save_path = f"../data/zines/{filename}"

    if response.status_code == 200:
        with open(save_path, "wb") as f:
            f.write(response.content)
        print(f"Downloaded: {filename}")
    else:
        print(f"Failed: {download_url} — status {response.status_code}")

    time.sleep(1)
    return filename

In [ ]:
# Download all 10 selected zines
# Each PDF is saved to data/zines/ with the format title_year.pdf
selected_urls = selected["download_url"].tolist()

downloaded_files = []
for url in selected_urls:
    filename = download_zine_auto(url)
    downloaded_files.append(filename)

print("\nAll downloads complete:")
for f in downloaded_files:
    print(" -", f)

In [ ]:
# Confirm the downloaded files are in the zines folder
zines_folder = "../data/zines"

all_files = os.listdir(zines_folder)
zine_files = [f for f in all_files if f.endswith(".pdf")]
zine_files.sort()

print("Zines ready for analysis:")
for f in zine_files:
    print(" -", f)
print("\nTotal zines:", len(zine_files))

## Building the Zine Dataframe

Each PDF filename follows the format `title_year.pdf` (for example, `behind_the_bars_2006.pdf`). I can extract the title and year automatically from the filename, without typing them in by hand.

This structure mirrors the Austen corpus used in class in `3-1_text_analysis_continued.ipynb`, where each row in a dataframe represented one novel. Here, each row represents one zine.

In [ ]:
# Build a dataframe with metadata extracted from filenames
# This uses an accumulator pattern: start with an empty list,
# then append a dictionary for each zine, then convert to a dataframe

zine_records = []

for filename in zine_files:
    # Remove .pdf to get something like "behind_the_bars_2006"
    name = filename.replace(".pdf", "")

    # The year is always the last 4 characters
    year = name[-4:]

    # The title is everything before the year, underscores replaced with spaces
    title = name[:-5].replace("_", " ")

    # Full path to the file
    filepath = os.path.join(zines_folder, filename)

    zine_records.append({
        "title": title,
        "year": year,
        "filepath": filepath
    })

zines_df = pd.DataFrame(zine_records)
print("Zines in dataframe:", len(zines_df))
zines_df

## Extracting Text from PDFs

I use `pdfplumber` to extract text from each zine page by page. Because zines often combine handwriting, unusual fonts, and image-heavy layouts, OCR quality can vary a lot.

Before running extraction on all 10 zines, I'll test it on one file first to make sure it's working and to see what the raw output looks like.

In [ ]:
# Test pdfplumber on the first zine before running all 10
test_path = zines_df["filepath"].iloc[0]
test_title = zines_df["title"].iloc[0]

test_text = ""

with pdfplumber.open(test_path) as pdf:
    for page in pdf.pages:
        page_text = page.extract_text()
        if page_text:
            test_text += page_text + " "

print("Test file:", test_title)
print("Characters extracted:", len(test_text))
print()
print("Preview (first 500 characters):")
print(test_text[:500])

The test output looks reasonable. Now I'll run the same extraction loop on all 10 zines and store the results in a new `full_text` column in the dataframe.

In [ ]:
# Extract text from each zine using pdfplumber
# Added per-page error handling so one problematic page doesn't freeze the whole loop

extracted_texts = []

for index, row in zines_df.iterrows():
    print("Extracting:", row["title"])

    full_text = ""
    failed_pages = 0

    try:
        with pdfplumber.open(row["filepath"]) as pdf:
            for page_num, page in enumerate(pdf.pages):
                try:
                    page_text = page.extract_text()
                    if page_text:
                        full_text += page_text + " "
                except Exception as e:
                    failed_pages += 1
                    print(f"  Skipping page {page_num + 1}: {e}")

    except Exception as e:
        print(f"  Could not open file: {e}")

    extracted_texts.append(full_text)
    print(f"  Characters extracted: {len(full_text)}")
    if failed_pages > 0:
        print(f"  Pages skipped due to errors: {failed_pages}")
    print("---")

zines_df["full_text"] = extracted_texts
print("\nExtraction complete!")

Three zines required intervention before analysis could proceed:

- **Easily Grossed Out #2**: the initial download produced a corrupt file (`No /Root object!`).
- **Gender Fuck Me** and **PATS #23**: extracted 0 characters because both are image-based zines with no text layer.

This is a concrete example of why zine selection requires human judgment, not just automated downloading. Whether a zine is computationally readable depends on how it was made. Digitally typeset zines extract cleanly, while handwritten or photocopied ones do not.

## Replacing Problem Zines

In [ ]:
# Identify which of the 10 selected zines had extraction problems
# (either 0 characters extracted, or a corrupt file)
failed_titles = zines_df[zines_df["full_text"].str.len() < 500]["title"].tolist()
print("Zines that failed extraction:")
print(failed_titles)

# Build a pool of replacement candidates from the catalog
# Exclude titles already selected, and filter to zines from 1990 onward
remaining = catalog_df[
    ~catalog_df["title"].isin(selected_titles) &
    (catalog_df["year_clean"] >= 1990)
].sample(30, random_state=99)

print("\nReplacement candidates to browse:")
print(remaining[["object_id", "title", "year"]].to_string(index=False))

In [ ]:
replacements = catalog_df[catalog_df["object_id"].isin([408, 337, 277])]
replacements[["object_id", "title", "year", "download_url"]]

In [ ]:
for url in replacements["download_url"].tolist():
    download_zine_auto(url)

In [ ]:
# Keep only the 10 final selected zines, dropping the 3 failed originals
final_titles = [
    "behind the bars",
    "doris 21",
    "empty orchestra 1",
    "gaysi 1",
    "gender trash 1",
    "going homo 3",
    "queer zine explosion 20",
    "the anarchistic queer 1.1",
    "timtum - a trans jew zine",
    "we have always existed transgender people through history"
]

zines_df = zines_df[zines_df["title"].isin(final_titles)].reset_index(drop=True)
print("Zines in dataframe:", len(zines_df))

## Text Quality Assessment

In [ ]:
# Preview the first 500 characters of each zine to assess text quality
for index, row in zines_df.iterrows():
    print("=== ", row["title"], "(", row["year"], ") ===")
    print("Total characters:", len(row["full_text"]))
    print("Preview:")
    print(row["full_text"][:500])
    print()

After inspecting the extracted text above, here are my observations for each zine:

- **Behind the Bars (2006)**: Good extraction with minor OCR errors (e.g., "arc" for "are", "pa11icipating" for "participating"). Usable for analysis.

- **Doris #21 (2003)**: Noisy extraction due to a heavily visual and handwritten layout. The preview shows broken words (e.g., "wri. te.", "Starti~", "ycm") and irregular spacing. Results should be interpreted with caution.

- **Empty Orchestra #1 (2012)**: Moderately noisy. The preview shows stray characters and formatting artifacts from the zine's visual layout (e.g., scattered punctuation marks and line breaks mid-word). Some content extracted usably, but noise is present throughout.

- **Gaysi #1 (2011)**: Clean extraction. The table of contents structure came through clearly, and the prose text that follows reads well. Good candidate for analysis.

- **Gender Trash #1 (1993)**: High character count (67,164) but the preview is almost entirely OCR noise. The decorative "gendertrash" text that runs along the border of the page was read as the primary content. The actual article text may be buried further in the document, but this zine's layout poses serious challenges for text extraction.

- **Going Homo #3 (1994)**: Good extraction overall with minor OCR artifacts (e.g., "gra fi.Jl" for "graffiti", "eitp" likely a truncated word). The prose content comes through clearly. Usable for analysis.

- **Queer Zine Explosion #20 (2003)**: Functional extraction. This is a review/compilation zine so the text is naturally fragmented — short blurbs rather than continuous prose. This will likely affect TF-IDF results, as distinctive terms may reflect the zines being reviewed rather than the zine's own voice.

- **The Anarchistic Queer #1.1 (2011)**: The preview is almost entirely garbled. What extracted appears to be OCR attempting to read handwritten or heavily stylized text. With only 15,678 characters and this level of noise, results for this zine are likely unreliable.

- **TimTum - A Trans Jew Zine (1999)**: Clean extraction with the highest character count in the sample (83,240). The prose text reads clearly. Strong candidate for TF-IDF analysis.

- **We Have Always Existed (2022)**: Very short extraction (7,146 characters). The text that did extract is clean and readable, but the low character count suggests most pages are image-based. Results should be treated as a partial picture of the zine's content.

## Text Preprocessing

I lowercase, remove punctuation, tokenize, and remove stop words. I also filter 
out single characters and non-alphabetical tokens to reduce OCR noise.

In [ ]:
# Extended punctuation list to catch curly quotes and dashes
# common in zine text that the standard punctuation list misses
extra_punctuation = "\u201c\u201d\u2018\u2019\u2014\u2013\u2022\u00b7"
all_punctuation = punctuation + extra_punctuation

def clean_and_tokenize(text):
    """
    Takes a raw text string and returns a list of cleaned tokens.
    Steps: lowercase → remove punctuation → split → remove stop words and noise.
    """
    # Step 1: lowercase
    clean_text = text.lower()

    # Step 2: remove punctuation
    for character in all_punctuation:
        clean_text = clean_text.replace(character, " ")

    # Step 3: tokenize by splitting on whitespace
    tokens = clean_text.split()

    # Steps 4 and 5: remove stop words, tokens shorter than 3 characters,
    # and non-alphabetical tokens
    # Minimum length raised from 2 to 3 to filter out OCR noise like "te"
    tokens = [t for t in tokens
          if t not in stop_words
          and len(t) > 2
          and len(t) < 25
          and t.isalpha()]

    return tokens

# Apply the function to every zine using .apply()
zines_df["cleaned_text"] = zines_df["full_text"].apply(clean_and_tokenize)
zines_df["word_count"] = zines_df["cleaned_text"].apply(len)

print("Word counts after cleaning:")
print(zines_df[["title", "word_count"]].to_string(index=False))

## Word Frequency Analysis
This addresses my first research question by identifying the most common words 
across all ten zines combined. High-frequency words reveal shared themes and 
vocabulary across the sample.

In [ ]:
# Word frequency across all zines combined
all_words = []
for tokens in zines_df["cleaned_text"]:
    all_words.extend(tokens)

overall_freq = nltk.FreqDist(all_words)

overall_freq_df = pd.DataFrame(
    overall_freq.most_common(25),
    columns=["word", "count"]
)

overall_freq_df

In [ ]:
# Plot top 25 most frequent words across all zines
overall_freq_df.plot(kind="bar", x="word", y="count", legend=False)

plt.title("Most Common Words Across All Zines")
plt.xlabel("Word")
plt.ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## TF-IDF Analysis
TF-IDF (Term Frequency-Inverse Document Frequency) identifies words that are 
distinctive to each individual zine — words that appear often in one zine but 
not across the others. This directly addresses my primary research question: 
what vocabulary most characterizes each zine?

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TfidfVectorizer needs plain strings, not token lists
# so I join each zine's tokens back into a single string
zines_df["cleaned_string"] = zines_df["cleaned_text"].apply(lambda x: " ".join(x))

vectorizer = TfidfVectorizer(max_features=5000)
tfidf_matrix = vectorizer.fit_transform(zines_df["cleaned_string"])

terms = vectorizer.get_feature_names_out()
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), index=zines_df["title"], columns=terms)

tfidf_df.shape

In [ ]:
# Get top distinctive words for each zine
def top_tfidf_terms(title, top_n=15):
    scores = tfidf_df.loc[title]
    scores = scores.sort_values(ascending=False)
    scores = scores.head(top_n)
    return pd.DataFrame({"term": scores.index, "tfidf": scores.values})

for title in tfidf_df.index:
    print("===", title, "===")
    print(top_tfidf_terms(title, top_n=10))
    print()

In [ ]:
# Plot top TF-IDF terms for all 10 zines in a 2-row grid
fig, axes = plt.subplots(nrows=2, ncols=5, figsize=(20, 10))

# axes is a 2D array (2 rows × 5 columns)
# so I flatten it into a simple list of 10 boxes
axes = axes.flatten()

for ax, title in zip(axes, tfidf_df.index):
    plot_df = top_tfidf_terms(title, top_n=10)
    plot_df = plot_df.sort_values("tfidf")
    ax.barh(plot_df["term"], plot_df["tfidf"])
    ax.set_title(title, fontsize=9)
    ax.set_xlabel("TF-IDF Score", fontsize=8)
    ax.tick_params(axis="y", labelsize=8)

fig.suptitle("Top 10 Distinctive Words by Zine", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Save extracted and cleaned zine data to CSV
zines_df[["title", "year", "full_text", "word_count"]].to_csv(
    "../output/extracted_text/zines_data.csv", 
    index=False
)
print("Data saved to output/extracted_text/zines_data.csv")

## Results

The TF-IDF grid above identifies the vocabulary most distinctive to each zine:
words frequent within one zine but rare across the others. Reading the subplots
together and grouping them by decade reveals how the language, politics, and
community concerns of queer zine culture shifted across thirty years.

### 1990s: Identity Formation and Subcultural Vocabulary

Three zines in the sample come from the 1990s: **Gender Trash #1 (1993)**,
**Going Homo #3 (1994)**, and **TimTum (1999)**.

This decade's zines show vocabulary oriented around the active construction
of queer and trans identity. The terminology for trans experience was still
being invented in this period, and Gender Trash's top TF-IDF terms reflect
that: language of gender identity that predates the mainstream vocabulary
shift of the 2000s. Going Homo's distinctive terms are subcultural: music,
scene, and community vocabulary tied to the Riot Grrrl-adjacent queer punk
milieu of the mid-1990s, where zines and underground music were inseparable.
TimTum, the longest document in the sample (83,240 characters), produces the
most stable TF-IDF results of any zine. Its top terms combine Hebrew- and
Yiddish-derived vocabulary with trans community language, a combination that
appears nowhere else in the corpus and marks it as occupying a very specific
intersectional position.

What unites the 1990s zines is that their vocabulary is constitutive:
language being used to name and build communities, not just describe them.

### 2000s: Political Registers and Established Discourse

The 2000s are represented by **Doris #21 (2003)**, **Queer Zine Explosion #20
(2003)**, and **Behind the Bars (2006)**.

By the early 2000s, queer and trans vocabulary had stabilized considerably,
and the TF-IDF terms for this decade's zines reflect more settled, though
diverse, political registers. Behind the Bars shows vocabulary consistent
with prison abolitionist politics, connecting LGBTQ experience to carceral
critique in a way that reflects the growing influence of abolitionism in queer
activist spaces during the mid-2000s. Doris #21 takes a very different
register: its top terms are personal and narrative, reflecting the confessional,
diaristic mode that defines the series. This contrast between Doris and Behind
the Bars, both from 2003, illustrates that queer zine vocabulary in this decade
was not uniform. Zines occupied a range of registers from the intimate to the
explicitly political.

Queer Zine Explosion #20 is a special case: as a review compilation, its
distinctive terms are largely proper nouns and vocabulary borrowed from the
zines it reviews rather than a voice of its own. Its TF-IDF results tell us
more about the zines being discussed than about QZE itself, a finding that
highlights how computational methods respond to form as much as content.

### 2010s: Intersectionality and Expanded Frames

**Gaysi #1 (2011)**, **The Anarchistic Queer #1.1 (2011)**, and **Empty
Orchestra #1 (2012)** represent the 2010s.

This decade's zines show the most explicitly intersectional vocabulary in the
sample. Gaysi #1 is the only zine oriented around South Asian queer identity,
and its distinctive terms reflect that dual positionality: vocabulary drawn
from LGBTQ discourse and South Asian cultural reference points that appear
nowhere else in the corpus. The Anarchistic Queer's top terms are the most
overtly ideological of any zine in the sample, linking queerness to anarchism
and anti-fascism in ways that reflect an intersectional politics increasingly
explicit in queer activist zines of this period. Empty Orchestra occupies a
different register: terms tied to performance, art, and DIY creative community,
consistent with the early 2010s moment when queer and trans organizing was
increasingly centered around cultural production.

The 2010s zines collectively suggest a vocabulary shift. Where 1990s zines
were building identity and 2000s zines were consolidating political discourse,
2010s zines are connecting queerness to other frameworks: national identity,
anarchism, artistic community. The vocabulary of this decade reaches outward
in ways the earlier decades do not.

### 2010s Onward: Intersectionality and Expanded Frames

The most recent zines in the sample span from 2011 to 2022: **Gaysi #1
(2011)**, **The Anarchistic Queer #1.1 (2011)**, **Empty Orchestra #1
(2012)**, and **We Have Always Existed (2022)**.

This group shows the most explicitly intersectional vocabulary in the sample.
Gaysi #1 is the only zine oriented around South Asian queer identity, and its
distinctive terms reflect that dual positionality: vocabulary drawn from LGBTQ
discourse and South Asian cultural reference points that appear nowhere else in
the corpus. The Anarchistic Queer's top terms are the most overtly ideological
of any zine in the sample, linking queerness to anarchism and anti-fascism in
ways that reflect an intersectional politics increasingly explicit in queer
activist zines of this period. Empty Orchestra occupies a different register:
terms tied to performance, art, and DIY creative community, consistent with the
early 2010s moment when queer and trans organizing was increasingly centered
around cultural production.

We Have Always Existed takes yet another direction. Its TF-IDF results
are dominated by the names of historical trans and gender-nonconforming figures,
a vocabulary pattern that reflects the zine's explicit archival project:
recovering and centering trans history. Where the 2010s zines connect queerness
outward to other political and cultural frameworks, this zine turns to the past,
written in the context of intensifying anti-trans legislation.

Taken together, these zines suggest a vocabulary shift from the earlier decades.
Where 1990s zines were building identity and 2000s zines were consolidating
political discourse, the zines from 2010 onward connect queerness to other
frameworks: national identity, anarchism, artistic community, and historical
memory. The vocabulary reaches outward and backward in ways the earlier decades
do not.

### Summary: Vocabulary Across the Decades

| Decade | Characteristic Vocabulary Register |
|--------|-------------------------------------|
| 1990s       | Identity construction, subcultural naming, community formation |
| 2000s       | Established political discourse, personal narrative, carceral critique |
| 2010s onward | Intersectionality, ideology, artistic community, diaspora, historical recovery |

These patterns are preliminary findings from a small, purposively selected
sample. They suggest that queer zine vocabulary is not static. The language
queer zines use shifts alongside the political and cultural contexts in which
they are made. TF-IDF, by surfacing what is distinctive within each document,
makes those shifts visible in a way that reading alone might not.

## Reflection

**What worked:**
- `pdfplumber` successfully extracted text from all 10 zines after replacing
  3 problem files
- Raising the minimum token length from 2 to 3 characters, and later adding
  a maximum length cap of 25 characters, progressively reduced OCR noise
  without discarding real words
- TF-IDF produced meaningful results across all 10 zines, with distinctive
  vocabulary that maps clearly onto each zine's subject matter, community,
  and moment
- The pipeline is modular: the same functions worked across zines from 1993
  to 2022 without modification

**What didn't work:**
- Several noise tokens survived preprocessing and appeared in TF-IDF results: `lhe` (Behind the Bars), `waa` (Empty Orchestra), `ani` (Doris 21), `tae` and `aad` (TimTum), `ofthe` (The Anarchistic Queer). These are OCR artifacts that passed the alphabetical filter because they happen to look like real words
- *Gender Trash #1* (1993) produced a long garbled token (`gendertrashgendertrash...`) from its decorative border text, which ranked highly in TF-IDF and required a manual fix to the preprocessing filter
- *We Have Always Existed* (671 words after cleaning) is too short for reliable TF-IDF results. With so few tokens, the scores reflect the names of a handful of historical figures rather than any broader vocabulary pattern
- Zines are not purely textual objects. They rely on visual art, handwriting, layout, and design that no text extraction tool can capture. This pipeline sees only a partial version of each zine

**What I learned:**
- Data cleaning took more time than the analysis itself. Three zines had to be replaced, preprocessing parameters had to be adjusted twice, and noise tokens still made it into the final results. This iterative process of finding and fixing problems is a normal part of computational research
- The decision about which zines are computationally readable is itself an interpretive act. A zine that cannot be processed becomes invisible to this kind of analysis, and that invisibility reflects the same tensions between standardization and difference that queer zine culture has always navigated
- Computational methods surface patterns but require human interpretation to be meaningful, which is why the analysis above does not stop at the TF-IDF scores but asks what those scores tell us about queer community, politics, and language across time
